# Quant Finance – Lernnotebook
# Modul 1: Marktgrundlagen & Renditeberechnung

**Autor:** Florian Ebner  
**Zweck:** Strukturiertes Selbststudium für Quant Risk / Quant Research Rollen

---

## Quellenverzeichnis dieses Moduls

- Markowitz, H. (1952). *Portfolio Selection*. Journal of Finance, 7(1), 77–91.
- Sharpe, W.F. (1964). *Capital Asset Prices: A Theory of Market Equilibrium*. Journal of Finance, 19(3), 425–442.
- Hull, J.C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*. McGraw-Hill.
- Fabozzi, F.J. (2007). *Fixed Income Analysis* (2nd ed.). CFA Institute.
- CFA Institute. (2020). *CFA Program Curriculum*. Level I–III.
- de Prado, M.L. (2018). *Advances in Financial Machine Learning*. Wiley.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.color': '#e0e0e0',
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
})
print('Setup complete. Modul 1 – Marktgrundlagen & Renditeberechnung')

---
## 1. Renditeberechnung: Täglich, Monatlich, Jährlich

### Theorie

Die Renditeberechnung ist der absolute Ausgangspunkt jeder quantitativen Analyse. Es gibt zwei grundlegende Arten:

**Simple Return (arithmetische Rendite):**
$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Log Return (stetige Rendite):**
$$r_t^{\log} = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

### Wann welche verwenden?

**Log Returns** sind in der Praxis Standard weil:
1. **Zeitadditivität:** Log-Renditen über T Perioden addieren sich: $r_{1\to T}^{\log} = \sum_{t=1}^T r_t^{\log}$. Arithmetische Renditen nicht.
2. **Normalverteilungsannahme:** Log-Renditen sind annähernd normalverteilt (streng genommen lognormal-verteilte Preise → normal-verteilte Log-Renditen). Das erlaubt den Einsatz parametrischer Statistik.
3. **Keine negativen Preise:** Da $\ln(P_t/P_{t-1})$ reelle Werte annehmen kann, sind Preise implizit immer positiv.

**Simple Returns** werden bevorzugt bei:
- Portfolio-Renditen (Gewichtete Summe von Simple Returns = Portfolio Simple Return)
- Kurzem Zeithorizont (Unterschied zu Log Return ist bei täglichen Renditen vernachlässigbar)

### Annualisierung

Um Renditen verschiedener Frequenzen vergleichbar zu machen, wird auf Jahresbasis skaliert:

$$r_{\text{ann}} = (1 + r_{\text{täglich}})^{252} - 1 \approx r_{\text{täglich}} \times 252 \quad (\text{näherungsweise})$$

**Handelstage:** In der Praxis werden 252 Handelstage pro Jahr verwendet (nicht 365 Kalendertage), da Märkte an Wochenenden und Feiertagen geschlossen sind.

**Quelle:** Hull (2018), Kap. 3; CFA Institute Curriculum Level I, Reading 7

In [ ]:
# ── Simuliere Aktienkurs (Geometrische Brownsche Bewegung) ────────────────────
# Realistisch: mu=8% p.a., sigma=20% p.a.
T        = 756          # 3 Jahre Handelstage
mu_ann   = 0.08
sig_ann  = 0.20
mu_d     = mu_ann  / 252
sig_d    = sig_ann / np.sqrt(252)
S0       = 100.0

log_returns_daily = np.random.normal(mu_d, sig_d, T)
prices = S0 * np.exp(np.cumsum(log_returns_daily))
prices = np.insert(prices, 0, S0)

# ── Renditeberechnung ─────────────────────────────────────────────────────────
simple_returns = pd.Series(prices).pct_change().dropna()
log_returns    = pd.Series(np.log(prices[1:] / prices[:-1]))

# ── Monatliche Renditen (Resampling) ──────────────────────────────────────────
dates      = pd.bdate_range('2021-01-01', periods=T+1)
price_ser  = pd.Series(prices, index=dates)
monthly_r  = price_ser.resample('ME').last().pct_change().dropna()

# ── Jährliche Renditen ────────────────────────────────────────────────────────
yearly_r   = price_ser.resample('YE').last().pct_change().dropna()

# ── Annualisierung ────────────────────────────────────────────────────────────
cagr = (prices[-1] / prices[0]) ** (252 / T) - 1   # Compound Annual Growth Rate
ann_vol  = simple_returns.std() * np.sqrt(252)

print(f'Gesamtrendite (Simple)       : {(prices[-1]/prices[0]-1)*100:.2f}%')
print(f'CAGR (annualisiert)          : {cagr*100:.2f}%')
print(f'Annualisierte Volatilität    : {ann_vol*100:.2f}%')
print(f'Mittelwert tägliche Rendite  : {simple_returns.mean()*100:.4f}%')
print(f'Differenz Simple vs Log Ret  : {(simple_returns.mean()-log_returns.mean())*10000:.4f} Basispunkte')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(dates, prices, color='#4a90d9', lw=1.5)
axes[0].set_title('Aktienkurs (simuliert)', fontweight='bold')
axes[0].set_ylabel('Preis (€)')

axes[1].hist(simple_returns * 100, bins=50, color='#4a90d9', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', lw=1, ls='--')
axes[1].set_title('Verteilung täglicher Renditen (%)', fontweight='bold')
axes[1].set_xlabel('Tagesrendite (%)')

axes[2].bar(range(len(yearly_r)), yearly_r * 100,
            color=['#2ecc71' if r > 0 else '#e74c3c' for r in yearly_r], edgecolor='white')
axes[2].set_xticks(range(len(yearly_r)))
axes[2].set_xticklabels([str(y.year) for y in yearly_r.index])
axes[2].set_title('Jährliche Renditen (%)', fontweight='bold')
axes[2].set_ylabel('Rendite (%)')
axes[2].axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.savefig('returns_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Volatilität

### Theorie

Volatilität ist das **wichtigste Risikomaß** in der Finanzwelt. Sie misst die Streuung der Renditen um ihren Erwartungswert.

**Historische Volatilität (Realisierte Volatilität):**
$$\sigma = \sqrt{\frac{1}{T-1} \sum_{t=1}^T (r_t - \bar{r})^2}$$

**Annualisierung:** $\sigma_{\text{ann}} = \sigma_{\text{täglich}} \times \sqrt{252}$ (Square-Root-of-Time Regel)

### Implizite Volatilität vs. Historische Volatilität

- **Historische Volatilität:** Rückwärtsblickend, berechnet aus vergangenen Preisen. Misst *was war*.
- **Implizite Volatilität (IV):** Aus Optionspreisen rückwärts berechnet via Black-Scholes. Misst *was der Markt erwartet*. Der **VIX** ist der bekannteste IV-Index (S&P 500, 30-Tage-Horizont).

### Volatilitäts-Clustering (ARCH/GARCH)

Empirisch beobachtet: Hohe Volatilität folgt auf hohe Volatilität, niedrige auf niedrige. Diese **Volatilitätscluster** werden durch **GARCH-Modelle** (Generalized AutoRegressive Conditional Heteroskedasticity) modelliert:

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

Ein einfaches GARCH(1,1): Die heutige Volatilität hängt von gestern's Schock ($\epsilon_{t-1}^2$) und gestern's Volatilität ($\sigma_{t-1}^2$) ab. Typische Parameter: $\alpha \approx 0.1$, $\beta \approx 0.85$.

### Fat Tails und Kurtosis

Reale Renditeverteilungen haben **dickere Enden (Fat Tails)** als die Normalverteilung – extreme Verluste treten häufiger auf als ein Normalverteilungsmodell vorhersagt. Gemessen durch **Excess Kurtosis** (Kurtosis - 3): Normalverteilung = 0, reale Aktien typisch 2–10.

**Quelle:** Engle, R. (1982). *Autoregressive Conditional Heteroscedasticity*. Econometrica, 50(4); Hull (2018), Kap. 23

In [ ]:
# ── Rollende Volatilität ──────────────────────────────────────────────────────
roll_vol_21  = simple_returns.rolling(21).std()  * np.sqrt(252) * 100  # 1-Monats-Fenster
roll_vol_63  = simple_returns.rolling(63).std()  * np.sqrt(252) * 100  # 3-Monats-Fenster
roll_vol_252 = simple_returns.rolling(252).std() * np.sqrt(252) * 100  # 1-Jahres-Fenster

# ── Kurtosis und Schiefe ──────────────────────────────────────────────────────
excess_kurtosis = simple_returns.kurtosis()    # scipy/pandas: excess kurtosis
skewness        = simple_returns.skew()

print(f'Annualisierte Volatilität (252-Tage): {roll_vol_252.dropna().iloc[-1]:.2f}%')
print(f'Excess Kurtosis                      : {excess_kurtosis:.4f}  (Normalverteilung = 0)')
print(f'Schiefe (Skewness)                   : {skewness:.4f}  (Normalverteilung = 0)')

# Jarque-Bera Normalitätstest
jb_stat, jb_p = stats.jarque_bera(simple_returns)
print(f'Jarque-Bera Test: stat={jb_stat:.2f}, p={jb_p:.6f} → ',
      'Nicht normalverteilt ✓' if jb_p < 0.05 else 'Normalverteilung nicht abgelehnt')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rollende Volatilität
axes[0].plot(dates[1:], roll_vol_21,  color='#e74c3c', lw=1,   alpha=0.8, label='21-Tage')
axes[0].plot(dates[1:], roll_vol_63,  color='#e67e22', lw=1.5, alpha=0.8, label='63-Tage')
axes[0].plot(dates[1:], roll_vol_252, color='#2c3e50', lw=2,   alpha=0.9, label='252-Tage')
axes[0].set_title('Rollende Annualisierte Volatilität', fontweight='bold')
axes[0].set_ylabel('Volatilität (% p.a.)')
axes[0].legend()

# Renditenverteilung vs. Normalverteilung
x = np.linspace(simple_returns.min(), simple_returns.max(), 300)
axes[1].hist(simple_returns, bins=60, density=True, color='#4a90d9',
             alpha=0.6, edgecolor='white', label='Empirisch')
axes[1].plot(x, stats.norm.pdf(x, simple_returns.mean(), simple_returns.std()),
             'r-', lw=2, label='Normalverteilung')
axes[1].set_title(f'Fat Tails: Kurtosis={excess_kurtosis:.2f}', fontweight='bold')
axes[1].legend()

# QQ-Plot
stats.probplot(simple_returns, dist='norm', plot=axes[2])
axes[2].set_title('QQ-Plot (Abweichung von Normalverteilung)', fontweight='bold')
axes[2].get_lines()[0].set(color='#4a90d9', markersize=2)

plt.tight_layout()
plt.savefig('volatility.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Korrelation

### Theorie

Die **Pearson-Korrelation** misst die lineare Abhängigkeit zwischen zwei Zeitreihen:
$$\rho_{XY} = \frac{\text{Cov}(X,Y)}{\sigma_X \cdot \sigma_Y} \in [-1, +1]$$

**Interpretation für Portfolios:**
- $\rho = +1$: Perfekt positiv korreliert – keine Diversifikation möglich
- $\rho = 0$: Keine lineare Beziehung – maximale Diversifikation
- $\rho = -1$: Perfekt negativ korreliert – vollständige Absicherung möglich

### Korrelationsinstabilität und Correlation Breakdown

Das gefährlichste Phänomen in der Risikomodellierung: In Krisenzeiten **steigen Korrelationen zwischen Aktien stark an**. Assets die normalerweise unabhängig sind (ρ ≈ 0.3), können in einem Crash gemeinsam fallen (ρ ≈ 0.9). Das macht Diversifikation genau dann unwirksam, wenn sie am meisten gebraucht wird.

Dokumentiert in: Longin & Solnik (2001), *Extreme Correlation of International Equity Markets*; auch im Crash von 2008 zu beobachten.

### Rollende Korrelation

Da Korrelationen zeitvariabel sind, ist die **rollende Korrelation** (berechnet über ein gleitendes Fenster) deutlich informativer als eine statische Schätzung über den gesamten Zeitraum.

### Rangkorrelation (Spearman)

**Spearman-Korrelation** misst monotone (nicht notwendig lineare) Beziehungen und ist robuster gegenüber Ausreißern. Wird oft in Factor-Research verwendet.

**Quelle:** Longin, F. & Solnik, B. (2001). Journal of Finance, 56(2), 649–676. Hull (2018), Kap. 9

In [ ]:
# ── Zwei Asset-Zeitreihen ─────────────────────────────────────────────────────
# Asset 1: höhere Rendite, höheres Risiko
r1 = pd.Series(np.random.normal(0.0004, 0.012, T))   # ~10% p.a., 19% Vol
# Asset 2: korreliert zu Asset 1 (ρ ≈ 0.6), niedrigeres Risiko
noise = np.random.normal(0, 0.009, T)
r2    = pd.Series(0.6 * r1 + np.sqrt(1 - 0.6**2) * noise)

# ── Statische Korrelation ─────────────────────────────────────────────────────
pearson_r, pearson_p   = stats.pearsonr(r1, r2)
spearman_r, spearman_p = stats.spearmanr(r1, r2)

print(f'Pearson-Korrelation  : {pearson_r:.4f}  (p={pearson_p:.2e})')
print(f'Spearman-Korrelation : {spearman_r:.4f}  (p={spearman_p:.2e})')

# ── Rollende Korrelation ──────────────────────────────────────────────────────
roll_corr_63  = r1.rolling(63).corr(r2)
roll_corr_126 = r1.rolling(126).corr(r2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(r1 * 100, r2 * 100, alpha=0.3, s=8, color='#4a90d9')
m, b = np.polyfit(r1, r2, 1)
x_fit = np.linspace(r1.min(), r1.max(), 100)
axes[0].plot(x_fit * 100, (m * x_fit + b) * 100, 'r-', lw=2)
axes[0].set_xlabel('Asset 1 Tagesrendite (%)')
axes[0].set_ylabel('Asset 2 Tagesrendite (%)')
axes[0].set_title(f'Streudiagramm: ρ = {pearson_r:.3f}', fontweight='bold')

axes[1].plot(roll_corr_63,  color='#e74c3c', lw=1.2, alpha=0.8, label='63-Tage (3M)')
axes[1].plot(roll_corr_126, color='#2c3e50', lw=2,   alpha=0.9, label='126-Tage (6M)')
axes[1].axhline(pearson_r, color='grey', ls='--', lw=1, label=f'Gesamt ρ={pearson_r:.3f}')
axes[1].axhline(0, color='black', lw=0.7)
axes[1].set_ylim(-1, 1)
axes[1].set_title('Rollende Korrelation (Instabilität!)', fontweight='bold')
axes[1].set_ylabel('Korrelation')
axes[1].legend()

plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Drawdown

### Theorie

**Drawdown** misst den Wertverlust eines Portfolios vom letzten Höchststand (Peak) bis zu einem Tiefstpunkt (Trough):

$$DD_t = \frac{V_t - \max_{s \leq t} V_s}{\max_{s \leq t} V_s}$$

**Maximum Drawdown (MDD):** Der größte beobachtete Drawdown über den gesamten Zeitraum:
$$MDD = \min_t DD_t$$

### Warum Drawdown oft wichtiger ist als Volatilität

Volatilität ist symmetrisch – sie bestraft Gewinne und Verluste gleich. Drawdown misst ausschließlich **negative Erfahrungen des Investors**. Psychologisch und regulatorisch ist MDD oft relevanter:
- Viele institutionelle Mandate haben **MDD-Limits**
- **Hedge Fund Redemptions:** Wenn ein Hedge Fund 20% unter seinem Höchststand liegt, beginnen institutionelle Investoren Kapital abzuziehen – ein selbstverstärkender Prozess
- **High Water Mark:** Viele Performance Fees sind an die Erholung über den letzten Peak gebunden

### Recovery Time (Underwater Period)

Wie lange dauert es, sich von einem Drawdown zu erholen? Das ist die **Underwater Period** – oft genauso wichtig wie die Drawdown-Tiefe selbst.

**Calmar Ratio:** $\text{Calmar} = \frac{\text{CAGR}}{|\text{MDD}|}$ – misst Rendite pro Einheit Drawdown-Risiko. Hedge Funds streben Calmar > 1 an.

**Quelle:** de Prado (2018), Kap. 14; Magdon-Ismail, M. & Atiya, A. (2004). *On the Maximum Drawdown of a Brownian Motion*. Journal of Applied Probability.

In [ ]:
# ── Kumulativer Reichtum ──────────────────────────────────────────────────────
cum_wealth = (1 + simple_returns).cumprod()

# ── Drawdown-Berechnung ───────────────────────────────────────────────────────
rolling_peak   = cum_wealth.cummax()
drawdown_series = (cum_wealth - rolling_peak) / rolling_peak * 100   # in %
mdd             = drawdown_series.min()
mdd_idx         = drawdown_series.idxmin()

# Recovery: wann wurde Peak nach MDD wiederhergestellt?
peak_before_mdd = rolling_peak.iloc[mdd_idx]
recovery_mask   = (cum_wealth.iloc[mdd_idx:] >= peak_before_mdd)
recovery_days   = recovery_mask.idxmax() if recovery_mask.any() else None

# ── Calmar Ratio ──────────────────────────────────────────────────────────────
calmar = cagr / abs(mdd / 100)

print(f'Maximum Drawdown   : {mdd:.2f}%')
print(f'MDD Datum          : Tag {mdd_idx}')
print(f'CAGR               : {cagr*100:.2f}%')
print(f'Calmar Ratio       : {calmar:.3f}')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Kumulativer Reichtum
axes[0].plot(dates[1:], cum_wealth, color='#4a90d9', lw=1.8, label='Portfolio Wert')
axes[0].plot(dates[1:], rolling_peak, color='#2c3e50', lw=1, ls='--', alpha=0.6, label='Rolling Peak')
axes[0].fill_between(dates[1:], cum_wealth, rolling_peak, alpha=0.15, color='#e74c3c')
axes[0].set_title('Kumulativer Portfoliowert & Drawdown', fontweight='bold')
axes[0].set_ylabel('Kumulativer Wert')
axes[0].legend()

# Drawdown
axes[1].fill_between(dates[1:], drawdown_series, 0, color='#e74c3c', alpha=0.5)
axes[1].plot(dates[1:], drawdown_series, color='#c0392b', lw=1)
axes[1].axhline(mdd, color='black', ls='--', lw=1.2, label=f'MDD = {mdd:.2f}%')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title(f'Drawdown-Verlauf  |  MDD = {mdd:.2f}%  |  Calmar = {calmar:.3f}', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Leverage und Margin

### Theorie

**Leverage** bedeutet, mehr Kapital einzusetzen als tatsächlich vorhanden – durch Kreditaufnahme oder Derivate. Ein Portfolio mit **2x Leverage** (200% investiert) hat doppelt so hohe Renditen *und* Verluste:

$$r_{\text{leveraged}} = L \cdot r_{\text{portfolio}} - (L-1) \cdot r_f$$

wobei $L$ der Leverage-Faktor und $r_f$ der Kreditkostensatz ist.

### Margin

**Initial Margin:** Das Kapital das beim Eingehen einer Position hinterlegt werden muss (typisch 10–50% des Positionswerts bei Futures).

**Maintenance Margin:** Mindestkapital das im Account verbleiben muss. Fällt das Account darunter → **Margin Call**: Der Broker fordert sofortige Nachschüsse.

**Margin Call:** Wenn sich der Markt gegen eine gehebelte Position bewegt und das Eigenkapital unter die Maintenance Margin fällt, muss der Investor entweder:
1. Kapital nachschießen, oder
2. Positionen zwangsweise liquidieren

Zwangsliquidierungen in Stressmärkten können **Firesale-Spiralen** auslösen: Viele gehebelte Investoren verkaufen gleichzeitig → Preise fallen weiter → mehr Margin Calls → noch mehr Verkäufe. Das war ein zentraler Mechanismus der Finanzkrise 2008.

### Leverage und Drawdown – die Asymmetrie

Leverage verstärkt Verluste überproportional wegen des **Compounding-Effekts**:
- Ein 50% Verlust benötigt +100% Gewinn zur Erholung
- 2x Leverage: Ein 25% Markteinbruch bedeutet 50% Verlust des Eigenkapitals
- 10x Leverage: Bereits ein 10% Markteinbruch = Totalverlust

**Quelle:** Hull (2018), Kap. 2; Brunnermeier, M. & Pedersen, L. (2009). *Market Liquidity and Funding Liquidity*. Review of Financial Studies

In [ ]:
# ── Leverage-Vergleich ────────────────────────────────────────────────────────
r_f_daily      = 0.035 / 252   # Kreditkosten = 3.5% p.a.
leverage_levels = [1, 1.5, 2, 3]
colors_lev      = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for L, col in zip(leverage_levels, colors_lev):
    r_lev = L * simple_returns - (L - 1) * r_f_daily
    cum   = (1 + r_lev).cumprod()
    axes[0].plot(dates[1:], cum, color=col, lw=1.8, label=f'{L}x Leverage')

    roll_pk  = cum.cummax()
    dd_lev   = (cum - roll_pk) / roll_pk * 100
    axes[1].plot(dates[1:], dd_lev, color=col, lw=1.5, alpha=0.85, label=f'{L}x: MDD={dd_lev.min():.1f}%')

axes[0].set_title('Kumulativer Wert bei verschiedenen Leverage-Stufen', fontweight='bold')
axes[0].set_ylabel('Kumulativer Wert')
axes[0].legend()

axes[1].fill_between(dates[1:], axes[1].get_lines()[-1].get_ydata(), 0,
                     alpha=0.05, color='red')
axes[1].set_title('Drawdown bei verschiedenen Leverage-Stufen', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('leverage.png', dpi=150, bbox_inches='tight')
plt.show()

print('Leverage-Analyse: MDD und Endwert im Vergleich')
for L, col in zip(leverage_levels, colors_lev):
    r_lev = L * simple_returns - (L - 1) * r_f_daily
    cum   = (1 + r_lev).cumprod()
    mdd_l = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    print(f'  {L}x Leverage: Endwert={cum.iloc[-1]:.3f}, MDD={mdd_l:.1f}%')

---
## 6. Gewichteter Durchschnitt der Portfolio-Renditen

### Theorie

Die **Portfolio-Rendite** ist der **gewichtete Durchschnitt** der Einzelasset-Renditen:

$$r_p = \sum_{i=1}^N w_i \cdot r_i = w^T r$$

wobei $w_i$ die Gewichte (summing to 1) und $r_i$ die Renditen der einzelnen Assets sind.

### Wichtige Eigenschaft

Das gilt **für einfache Renditen** (Simple Returns) exakt. Für Log-Renditen gilt es nur näherungsweise. Das ist ein Grund warum Simple Returns in der Portfolio-Berechnung bevorzugt werden, obwohl Log-Renditen für Zeitreihenanalyse besser geeignet sind.

### Rebalancing und Drift

Über die Zeit verändert sich die Gewichtung durch unterschiedliche Renditen (**Drift**). Ein Portfolio das am Anfang 50/50 auf zwei Assets aufgeteilt war, kann nach einem Jahr 70/30 sein wenn ein Asset stark gestiegen ist. **Rebalancing** setzt die Gewichte periodisch auf die Zielallokation zurück – das erzwingt systematisch Buy Low / Sell High und hat empirisch einen kleinen positiven Renditeeffekt.
**Quelle:** Grinold & Kahn (1999), Kap. 2

In [ ]:
# ── 4-Asset-Portfolio ─────────────────────────────────────────────────────────
n_a   = 4
names = ['SAP', 'Allianz', 'Siemens', 'RWE']
vols  = [0.22, 0.18, 0.20, 0.16]
mus   = [0.10, 0.08, 0.09, 0.06]
corr4 = np.array([[1,.4,.5,.2],[.4,1,.5,.3],[.5,.5,1,.3],[.2,.3,.3,1]])

# Simuliere tägliche Renditen
d_vols = np.array(vols) / np.sqrt(252)
d_mus  = np.array(mus)  / 252
cov4   = np.outer(d_vols, d_vols) * corr4
ret4   = np.random.multivariate_normal(d_mus, cov4, T)
df4    = pd.DataFrame(ret4, columns=names)

# Gewichte
w = np.array([0.35, 0.25, 0.25, 0.15])

# ── Buy-and-Hold vs. Monatliches Rebalancing ──────────────────────────────────
# Buy and Hold: Gewichte driften
port_bnh  = (df4 * w).sum(axis=1)           # tägliche Portfolio-Rendite
cum_bnh   = (1 + port_bnh).cumprod()

# Monatliches Rebalancing (~21 Handelstage)
cum_reb   = pd.Series(np.zeros(T), dtype=float)
wealth    = 1.0
cur_w     = w.copy()
for t in range(T):
    if t % 21 == 0 and t > 0:
        cur_w = w.copy()    # rebalance back to target
    r_t = (df4.iloc[t].values * cur_w).sum()
    wealth *= (1 + r_t)
    cur_w  *= (1 + df4.iloc[t].values)
    cur_w  /= cur_w.sum()
    cum_reb.iloc[t] = wealth

print('Buy-and-Hold vs. Monatliches Rebalancing:')
ann_bnh = cum_bnh.iloc[-1] ** (252/T) - 1
ann_reb = cum_reb.iloc[-1] ** (252/T) - 1
print(f'  Buy-and-Hold CAGR    : {ann_bnh*100:.2f}%')
print(f'  Rebalancing CAGR     : {ann_reb*100:.2f}%')
print(f'  Rebalancing Bonus    : {(ann_reb-ann_bnh)*100:.3f}%')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates[1:], cum_bnh, color='#4a90d9', lw=2, label=f'Buy & Hold (CAGR={ann_bnh*100:.1f}%)')
ax.plot(dates[1:], cum_reb, color='#2ecc71', lw=2, ls='--', label=f'Monatl. Rebalancing (CAGR={ann_reb*100:.1f}%)')
ax.set_title('Portfolio: Buy & Hold vs. Monatliches Rebalancing', fontweight='bold')
ax.set_ylabel('Kumulativer Wert')
ax.legend()
plt.tight_layout()
plt.savefig('rebalancing.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Beta, Alpha und Faktor-Exposures

### Theorie

**Beta** misst die **Sensitivität eines Assets gegenüber einem Marktindex** (typisch: S&P 500, DAX):

$$\beta = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)} = \rho_{im} \cdot \frac{\sigma_i}{\sigma_m}$$

- $\beta = 1$: Asset bewegt sich wie der Markt
- $\beta > 1$: Aggressiv – übertrifft den Markt in beide Richtungen (z.B. Tech-Aktien)
- $\beta < 1$: Defensiv – schwächt Marktbewegungen ab (z.B. Utilities)
- $\beta < 0$: Inverse Bewegung zum Markt (z.B. Gold, manche Bonds)

**Alpha (Jensen's Alpha)** misst die **risikobereinigte Überrendite** die *nicht* durch Marktrisiko erklärt wird:

$$\alpha = r_i - [r_f + \beta (r_m - r_f)]$$

Positives Alpha bedeutet, dass ein Manager oder eine Strategie **mehr liefert als durch das eingegangene Marktrisiko gerechtfertigt**. In der Praxis ist dauerhaftes positives Alpha sehr selten und Gegenstand erbitterter Debatten (Efficient Market Hypothesis).

### Faktor-Exposure

Beta ist nur der einfachste Faktor. Faktoren sind systematische Quellen von Rendite und Risiko:
- **Market Factor (MKT):** Beta zum Gesamtmarkt
- **Size (SMB):** Small-cap vs. Large-cap Prämie (Fama-French)
- **Value (HML):** Value vs. Growth Prämie (Fama-French)
- **Momentum (MOM):** Gewinner tendieren kurzfristig zu weiteren Gewinnen (Carhart)
- **Qualität, Low Volatility, Profitabilität:** Neuere Faktoren

Die Faktor-Regression schätzt die Exposures:
$$r_i - r_f = \alpha + \beta_1 \cdot MKT + \beta_2 \cdot SMB + \beta_3 \cdot HML + \beta_4 \cdot MOM + \epsilon$$

**Quelle:** Fama, E. & French, K. (1993). *Common Risk Factors in the Returns on Stocks and Bonds*. Journal of Financial Economics, 33(1); Carhart, M. (1997). Journal of Finance, 52(1)

In [ ]:
# ── Simuliere Marktrenditen und Einzelaktie ───────────────────────────────────
true_beta  = 1.35
true_alpha = 0.0001   # leicht positives tägliches Alpha

r_market   = pd.Series(np.random.normal(0.0003, 0.011, T))   # Marktrenditen
r_specific = pd.Series(np.random.normal(0, 0.008, T))         # idiosynkratisches Risiko
r_rf_d     = 0.035 / 252

# Erzeuge Aktienrenditen via CAPM-Struktur
r_stock    = r_rf_d + true_alpha + true_beta * (r_market - r_rf_d) + r_specific

# ── OLS-Regression: Beta und Alpha schätzen ───────────────────────────────────
X    = r_market.values - r_rf_d
y    = r_stock.values  - r_rf_d
beta_hat, alpha_hat, r_value, p_value, std_err = stats.linregress(X, y)

# R² (Erklärungsgrad: wie viel Varianz durch Markt erklärt?)
r_squared = r_value ** 2

print(f'Wahres   Beta / Alpha  : {true_beta:.3f} / {true_alpha:.6f}')
print(f'Geschätzte Beta / Alpha: {beta_hat:.3f} / {alpha_hat:.6f}')
print(f'R²                     : {r_squared:.4f}  ({r_squared*100:.1f}% durch Markt erklärt)')
print(f'Annualisiertes Alpha   : {alpha_hat*252*100:.3f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Aktie vs. Markt
axes[0].scatter((r_market - r_rf_d) * 100, (r_stock - r_rf_d) * 100,
                alpha=0.2, s=8, color='#4a90d9')
x_fit = np.linspace(X.min(), X.max(), 100)
axes[0].plot(x_fit * 100, (alpha_hat + beta_hat * x_fit) * 100, 'r-', lw=2,
             label=f'β={beta_hat:.3f}, α={alpha_hat*252*100:.3f}% p.a.')
axes[0].set_xlabel('Markt Excess Return (%)')
axes[0].set_ylabel('Aktie Excess Return (%)')
axes[0].set_title(f'Beta-Regression  |  R² = {r_squared:.3f}', fontweight='bold')
axes[0].legend()

# Rollende Beta
roll_beta = pd.Series([
    stats.linregress(r_market.iloc[max(0,i-63):i].values - r_rf_d,
                     r_stock.iloc[max(0,i-63):i].values - r_rf_d)[0]
    for i in range(63, T)
])
axes[1].plot(roll_beta.values, color='#4a90d9', lw=1.5)
axes[1].axhline(true_beta, color='red', ls='--', lw=1.5, label=f'Wahres Beta={true_beta}')
axes[1].axhline(1.0,       color='grey', ls=':', lw=1, label='Beta=1 (Markt)')
axes[1].set_title('Rollende Beta (63-Tage-Fenster)', fontweight='bold')
axes[1].set_ylabel('Beta')
axes[1].legend()

plt.tight_layout()
plt.savefig('beta_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Fama-French & Carhart Faktormodelle

### Theorie

Das **CAPM** erklärt Renditen durch einen einzigen Faktor (Marktrisiko). Empirisch zeigen Fama und French, dass zwei weitere Faktoren systematisch Renditen erklären:

**Fama-French 3-Faktor-Modell (1993):**
$$r_i - r_f = \alpha + \beta_{MKT} \cdot MKT + \beta_{SMB} \cdot SMB + \beta_{HML} \cdot HML + \epsilon$$

- **SMB (Small Minus Big):** Rendite von Small-Cap-Portfolios minus Large-Cap-Portfolios. Historisch ~3% p.a. Prämie.
- **HML (High Minus Low):** Rendite von Value-Aktien (hohe Buch-/Marktwert-Ratio) minus Growth-Aktien. Historisch ~4% p.a. Prämie.

**Carhart 4-Faktor-Modell (1997):** Ergänzt um:
- **MOM (Momentum):** Rendite der letzten 12 Monate (minus letzter Monat) als Prädiktor. Aktien die gut gelaufen sind tendieren kurzfristig weiter gut zu laufen.

**Fama-French 5-Faktor-Modell (2015):** Ergänzt um:
- **RMW (Robust Minus Weak):** Profitabilitätsprämie
- **CMA (Conservative Minus Aggressive):** Investitionsprämie

### Bedeutung für Risk Quants

Faktormodelle sind das Standard-Werkzeug für **Risiko-Attribution**: Wieviel des Portfoliorisikos kommt von Marktrisiko, Size-Tilt, Value-Tilt etc.? Ein Portfolio mit hohem SMB-Exposure ist anfällig für Small-Cap-Drawdowns – auch wenn das dem Portfolio-Manager nicht bewusst ist.

**Datenbezug:** Fama-French Faktoren sind kostenlos unter https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html verfügbar.

**Quelle:** Fama & French (1993, 2015); Carhart (1997)

In [ ]:
# ── Simulierte Faktor-Renditen (realistisch kalibriert) ───────────────────────
MKT = pd.Series(np.random.normal(0.0003, 0.011, T))     # ~7.5% p.a., 17% vol
SMB = pd.Series(np.random.normal(0.00012, 0.006, T))    # ~3% p.a., 10% vol
HML = pd.Series(np.random.normal(0.00016, 0.007, T))    # ~4% p.a., 11% vol
MOM = pd.Series(np.random.normal(0.00020, 0.010, T))    # ~5% p.a., 16% vol

# Echte Exposures des Portfolios (werden geschätzt)
true_betas = {'MKT': 0.95, 'SMB': 0.40, 'HML': 0.20, 'MOM': 0.15}
true_alpha_ff = 0.00005

# Portfolio-Renditen
r_port_ff = (true_alpha_ff
             + true_betas['MKT'] * MKT
             + true_betas['SMB'] * SMB
             + true_betas['HML'] * HML
             + true_betas['MOM'] * MOM
             + pd.Series(np.random.normal(0, 0.005, T)))  # idiosynkratisch

# ── OLS Regression: 4-Faktor-Modell ──────────────────────────────────────────
factors = pd.DataFrame({'MKT': MKT, 'SMB': SMB, 'HML': HML, 'MOM': MOM})
X_ff    = np.column_stack([np.ones(T), factors.values])
y_ff    = r_port_ff.values

# Least Squares
coefs, residuals, rank, sv = np.linalg.lstsq(X_ff, y_ff, rcond=None)
alpha_ff_est = coefs[0]
beta_est     = dict(zip(['MKT', 'SMB', 'HML', 'MOM'], coefs[1:]))

y_hat    = X_ff @ coefs
ss_res   = np.sum((y_ff - y_hat) ** 2)
ss_tot   = np.sum((y_ff - y_ff.mean()) ** 2)
r2_ff    = 1 - ss_res / ss_tot

print('Carhart 4-Faktor-Modell – Ergebnisse:')
print(f'  Alpha (ann.)   : {alpha_ff_est*252*100:.3f}%')
print(f'  R²             : {r2_ff:.4f}')
print(f'  Faktor-Exposures:')
for fname, true_b in true_betas.items():
    print(f'    {fname}: geschätzt={beta_est[fname]:.3f}, wahres={true_b:.3f}')

# Visualisierung: Faktor-Beiträge zur Rendite
fig, ax = plt.subplots(figsize=(8, 5))
factor_contributions = {f: beta_est[f] * factors[f].mean() * 252 * 100
                        for f in ['MKT', 'SMB', 'HML', 'MOM']}
factor_contributions['Alpha'] = alpha_ff_est * 252 * 100

cols  = ['#e74c3c' if v < 0 else '#2ecc71' for v in factor_contributions.values()]
bars  = ax.bar(factor_contributions.keys(), factor_contributions.values(), color=cols, edgecolor='white')
ax.bar_label(bars, fmt='%.3f%%', padding=3)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Beitrag zur ann. Rendite (%)')
ax.set_title('Faktor-Attribution: Carhart 4-Faktor-Modell', fontweight='bold')
plt.tight_layout()
plt.savefig('factor_model.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Zusammenfassung Modul 1

| Konzept | Kernformel | Regulatorische / Praktische Relevanz |
|---------|-----------|--------------------------------------|
| **Log vs. Simple Return** | $r^{log} = \ln(P_t/P_{t-1})$ | Log additiv über Zeit; Simple für Portfoliogewichtung |
| **Volatilität** | $\sigma_{ann} = \sigma_d \times \sqrt{252}$ | Basisgröße für VaR, Margin, Optionspreise |
| **Korrelation** | $\rho = \text{Cov}/(\sigma_i \sigma_j)$ | Diversifikationsgrad; Correlation Breakdown in Krisen |
| **Drawdown** | $DD_t = (V_t - \text{Peak}_t)/\text{Peak}_t$ | MDD-Limits in Mandaten; Calmar Ratio |
| **Leverage** | $r_{lev} = L \cdot r - (L-1) r_f$ | Margin Calls; Firesale-Spiralen |
| **Portfolio Return** | $r_p = w^T r$ | Grundlage jeder Portfolio-Berechnung |
| **Beta** | $\beta = \text{Cov}(r_i,r_m)/\text{Var}(r_m)$ | Marktrisiko-Exposure; CAPM |
| **Alpha** | $\alpha = r_i - [r_f + \beta(r_m-r_f)]$ | Mehrwert über Marktbeta hinaus |
| **Fama-French** | $r_i - r_f = \alpha + \beta_{MKT} + \beta_{SMB} + \beta_{HML}$ | Risiko-Attribution; Faktor-Investing |

---
**→ Weiter mit Modul 2: Portfoliotheorie, CAPM, Efficient Frontier**